In [1]:
import numpy as np
import pandas as pd
import faiss

from pathlib import Path

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("FAISS:", faiss.__version__)

NumPy: 1.26.4
Pandas: 2.2.2
FAISS: 1.15.1


Set project paths and load embeddings

In [2]:
PROJECT_ROOT = Path.cwd().parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "runs" / "models"
REPORT_DIR = PROJECT_ROOT / "runs" / "reports"

REPORT_DIR.mkdir(parents=True, exist_ok=True)

user_embeddings = np.load(
    MODEL_DIR / "user_embeddings.npy"
).astype("float32")

movie_embeddings = np.load(
    MODEL_DIR / "movie_embeddings.npy"
).astype("float32")

train_df = pd.read_csv(
    PROCESSED_DIR / "train.csv"
)

movie_mapping = pd.read_csv(
    PROCESSED_DIR / "movie_mapping.csv"
)

movies_df = pd.read_csv(
    PROJECT_ROOT / "data" / "ml-latest-small" / "movies.csv"
)

print("User embeddings:", user_embeddings.shape)
print("Movie embeddings:", movie_embeddings.shape)
print("Train data:", train_df.shape)
print("Movie mapping:", movie_mapping.shape)

User embeddings: (610, 64)
Movie embeddings: (9742, 64)
Train data: (80419, 6)
Movie mapping: (9742, 2)


Normalize movie embeddings

In [3]:
movie_embeddings_normalized = movie_embeddings.copy()

faiss.normalize_L2(movie_embeddings_normalized)

print("Movie embeddings normalized.")
print("Shape:", movie_embeddings_normalized.shape)

print(
    "First 5 vector norms:",
    np.linalg.norm(movie_embeddings_normalized[:5], axis=1)
)

Movie embeddings normalized.
Shape: (9742, 64)
First 5 vector norms: [0.99999994 0.99999994 1.         1.0000001  1.        ]


Create the FAISS index

In [4]:
embedding_dimension = movie_embeddings_normalized.shape[1]

faiss_index = faiss.IndexFlatIP(embedding_dimension)

faiss_index.add(movie_embeddings_normalized)

print("FAISS index created successfully.")
print("Total indexed movies:", faiss_index.ntotal)
print("Embedding dimension:", faiss_index.d)

FAISS index created successfully.
Total indexed movies: 9742
Embedding dimension: 64


Prepare user embeddings for search

In [5]:
user_embeddings_normalized = user_embeddings.copy()

faiss.normalize_L2(user_embeddings_normalized)

print("User embeddings normalized.")
print("Shape:", user_embeddings_normalized.shape)

User embeddings normalized.
Shape: (610, 64)


Retrieve top-100 movie candidates for one user

In [6]:
TOP_K = 100
SEARCH_K = 200

sample_user_idx = 0

query_vector = user_embeddings_normalized[
    sample_user_idx:sample_user_idx + 1
]

scores, movie_indices = faiss_index.search(
    query_vector,
    SEARCH_K
)

candidate_movie_indices = movie_indices[0]
candidate_scores = scores[0]

print("User index:", sample_user_idx)
print("Retrieved candidates:", len(candidate_movie_indices))

print("\nTop 10 retrieved movie indices:")
print(candidate_movie_indices[:10])

print("\nTop 10 similarity scores:")
print(candidate_scores[:10])

User index: 0
Retrieved candidates: 200

Top 10 retrieved movie indices:
[2173 1288 1595 2696 1109 1192 1246 3661 4045 3723]

Top 10 similarity scores:
[-0.17101231 -0.182374   -0.18470624 -0.19025998 -0.19084707 -0.1958697
 -0.19965263 -0.20013346 -0.20021236 -0.20172071]


Remove previously watched movies

In [7]:
seen_movies = (
    train_df.groupby("user_idx")["movie_idx"]
    .apply(set)
    .to_dict()
)

user_seen_movies = seen_movies.get(
    sample_user_idx,
    set()
)

filtered_candidates = []

for movie_idx, score in zip(
    candidate_movie_indices,
    candidate_scores
):
    if movie_idx == -1:
        continue

    if movie_idx not in user_seen_movies:
        filtered_candidates.append((int(movie_idx), float(score)))

    if len(filtered_candidates) == TOP_K:
        break

print("Movies watched by user:", len(user_seen_movies))
print("Unwatched candidates retrieved:", len(filtered_candidates))

Movies watched by user: 185
Unwatched candidates retrieved: 100


Create recommendation dataframe

In [8]:
candidate_df = pd.DataFrame(
    filtered_candidates,
    columns=["movie_idx", "similarity_score"]
)

candidate_df.insert(
    0,
    "user_idx",
    sample_user_idx
)

candidate_df.insert(
    1,
    "rank",
    range(1, len(candidate_df) + 1)
)

display(candidate_df.head(10))

,user_idx,rank,movie_idx,similarity_score
0,0,1,2173,-0.171012
1,0,2,1288,-0.182374
2,0,3,1109,-0.190847
3,0,4,1192,-0.195870
4,0,5,1246,-0.199653
5,0,6,3661,-0.200133
6,0,7,4045,-0.200212
7,0,8,3723,-0.201721
8,0,9,3702,-0.202818
9,0,10,2131,-0.203948


Convert movie indices into movie titles

In [9]:
movie_lookup = movie_mapping.merge(
    movies_df,
    left_on="movieId",
    right_on="movieId",
    how="left"
)

sample_recommendations = candidate_df.merge(
    movie_lookup[["movie_idx", "title", "genres"]],
    on="movie_idx",
    how="left"
)

display(
    sample_recommendations[
        ["user_idx", "rank", "title", "genres", "similarity_score"]
    ].head(10)
)

,user_idx,rank,title,genres,similarity_score
0,0,1,"Mystery, Alaska (1999)",Comedy|Drama,-0.171012
1,0,2,Scream 2 (1997),Comedy|Horror|Mystery|Thriller,-0.182374
2,0,3,Prefontaine (1997),Drama,-0.190847
3,0,4,Cop Land (1997),Action|Crime|Drama|Thriller,-0.195870
4,0,5,Swept from the Sea (1997),Drama|Romance,-0.199653
5,0,6,Dragonslayer (1981),Action|Adventure|Fantasy,-0.200133
6,0,7,Galaxy of Terror (Quest) (1981),Action|Horror|Mystery|Sci-Fi,-0.200212
7,0,8,40 Days and 40 Nights (2002),Comedy|Romance,-0.201721
8,0,9,Hart's War (2002),Drama|War,-0.202818
9,0,10,Outside Providence (1999),Comedy,-0.203948


Generate candidates for all users

In [10]:
TOP_K = 100
SEARCH_K = min(1000, faiss_index.ntotal)

all_user_queries = user_embeddings_normalized.astype("float32")

all_scores, all_indices = faiss_index.search(
    all_user_queries,
    SEARCH_K
)

all_candidate_rows = []

for user_idx in range(len(user_embeddings)):

    user_seen_movies = seen_movies.get(user_idx, set())

    user_candidates = []

    for movie_idx, score in zip(
        all_indices[user_idx],
        all_scores[user_idx]
    ):

        if movie_idx == -1:
            continue

        if movie_idx in user_seen_movies:
            continue

        user_candidates.append(
            (int(movie_idx), float(score))
        )

        if len(user_candidates) == TOP_K:
            break

    for rank, (movie_idx, score) in enumerate(
        user_candidates,
        start=1
    ):
        all_candidate_rows.append({
            "user_idx": user_idx,
            "rank": rank,
            "movie_idx": movie_idx,
            "similarity_score": score
        })

all_candidates_df = pd.DataFrame(all_candidate_rows)

print("Candidate generation completed.")
print("Total candidates:", len(all_candidates_df))
print("Users:", all_candidates_df["user_idx"].nunique())

display(all_candidates_df.head(10))

Candidate generation completed.
Total candidates: 61000
Users: 610


,user_idx,rank,movie_idx,similarity_score
0,0,1,2173,-0.171012
1,0,2,1288,-0.182374
2,0,3,1109,-0.190847
3,0,4,1192,-0.195870
4,0,5,1246,-0.199653
5,0,6,3661,-0.200133
6,0,7,4045,-0.200212
7,0,8,3723,-0.201721
8,0,9,3702,-0.202818
9,0,10,2131,-0.203948


Add movie titles to all candidates

In [11]:
all_candidates_df = all_candidates_df.merge(
    movie_lookup[["movie_idx", "title", "genres"]],
    on="movie_idx",
    how="left"
)

display(
    all_candidates_df[
        ["user_idx", "rank", "title", "genres", "similarity_score"]
    ].head(20)
)

,user_idx,rank,title,genres,similarity_score
0,0,1,"Mystery, Alaska (1999)",Comedy|Drama,-0.171012
1,0,2,Scream 2 (1997),Comedy|Horror|Mystery|Thriller,-0.182374
2,0,3,Prefontaine (1997),Drama,-0.190847
3,0,4,Cop Land (1997),Action|Crime|Drama|Thriller,-0.195870
4,0,5,Swept from the Sea (1997),Drama|Romance,-0.199653
5,0,6,Dragonslayer (1981),Action|Adventure|Fantasy,-0.200133
6,0,7,Galaxy of Terror (Quest) (1981),Action|Horror|Mystery|Sci-Fi,-0.200212
7,0,8,40 Days and 40 Nights (2002),Comedy|Romance,-0.201721
8,0,9,Hart's War (2002),Drama|War,-0.202818
9,0,10,Outside Providence (1999),Comedy,-0.203948


Save candidate generation results

In [12]:
candidate_output_path = (
    REPORT_DIR / "faiss_candidate_recommendations.csv"
)

all_candidates_df.to_csv(
    candidate_output_path,
    index=False
)

print("FAISS candidate recommendations saved:")
print(candidate_output_path)

print("\nFile size (KB):",
      round(candidate_output_path.stat().st_size / 1024, 2))

FAISS candidate recommendations saved:
c:\Users\yatha\Desktop\personalized-recommendation-engine\runs\reports\faiss_candidate_recommendations.csv

File size (KB): 4630.03


Verify recommendation coverage

In [13]:
candidates_per_user = (
    all_candidates_df.groupby("user_idx")
    .size()
)

print("Minimum candidates per user:", candidates_per_user.min())
print("Maximum candidates per user:", candidates_per_user.max())
print("Average candidates per user:", round(candidates_per_user.mean(), 2))

print("\nUsers with 100 candidates:",
      (candidates_per_user == 100).sum())

print("Total users:", len(candidates_per_user))

Minimum candidates per user: 100
Maximum candidates per user: 100
Average candidates per user: 100.0

Users with 100 candidates: 610
Total users: 610
